# Lab 3: White-box distillation

In Lab 2 you measured a baseline and a teacher. Now you will train a student on the teacher's
probabilities and measure how much of the gap it closes.

You will:

1. Score every message with the teacher and save the scores (the slow step, done once).
2. Train a student on the teacher's **top answers**.
3. Train a student on the teacher's **probabilities**.
4. Compare both with the baseline and the teacher.
5. Choose a temperature on the validation set.

This lab needs the results saved by Lab 2. Allow about 45 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment
from kd_course.results import require_result

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

baseline_result = require_result("baseline_gold_616")

## 1. The data

The same four piles as in Lab 2.

In [ ]:
from kd_course.data import load_banking77, make_splits

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)

## 2. Score every message with the teacher

For each message the teacher gives a probability to each of the 77 types. The scores are saved in the
results folder. If you run this cell again, the saved files are read and the teacher is not run.

In [ ]:
from kd_course.label_scoring import score_or_load
from kd_course.models import TEACHER_ID, load_causal_lm
from kd_course.results import default_results_dir

teacher, teacher_tokenizer = load_causal_lm(TEACHER_ID)
folder = default_results_dir() / "teacher_scores"
args = {
    "model": teacher,
    "tokenizer": teacher_tokenizer,
    "label_names": LABELS,
    "progress_every": 500,
}

test_scores = score_or_load(folder / "test.json", texts=splits.test.texts, **args)
gold_scores = score_or_load(folder / "gold_train.json", texts=splits.gold_train.texts, **args)
pool_scores = score_or_load(folder / "pool.json", texts=splits.pool.texts, **args)

In [ ]:
import gc

import torch

del teacher, args
gc.collect()
torch.cuda.empty_cache()

## 3. The teacher's score from its probabilities

Take the teacher's most likely type for each test message. This is the ceiling for this lab.

In [ ]:
from kd_course.label_scoring import top_labels
from kd_course.metrics import summarize
from kd_course.results import save_result

teacher_pred = top_labels(test_scores)
teacher_result = summarize(splits.test.labels, teacher_pred, N)
save_result("teacher_scored", teacher_result, teacher_pred)
print(teacher_result)

## 4. A student trained on the teacher's top answers

The pool messages get the teacher's top choice as if it were a label. The 616 gold messages keep
their human labels. This is ordinary fine-tuning; only the source of most labels has changed.

In [ ]:
from kd_course.classifier import STUDENT_ENCODER_ID, predict, train_classifier
from kd_course.data import LabelledSet

hard_train = LabelledSet(
    texts=splits.gold_train.texts + splits.pool.texts,
    labels=splits.gold_train.labels + top_labels(pool_scores),
)
hard_student = train_classifier(
    STUDENT_ENCODER_ID, hard_train, splits.gold_val, num_labels=N, epochs=5, batch_size=32
)
hard_pred = predict(hard_student, splits.test.texts)
hard_result = summarize(splits.test.labels, hard_pred, N)
save_result("distilled_top_answers", hard_result, hard_pred)
print(hard_result)

del hard_student
gc.collect()
torch.cuda.empty_cache()

## 5. A student trained on the teacher's probabilities

Same messages, same teacher. This time the student sees the whole distribution, through the
distillation loss from lesson 3.4. Pool messages have no human label (`None`).

In [ ]:
from kd_course.distill import DistillSet, train_distilled_classifier

soft_train = DistillSet(
    texts=splits.gold_train.texts + splits.pool.texts,
    teacher_log_probs=gold_scores + pool_scores,
    labels=splits.gold_train.labels + [None] * len(splits.pool),
)
soft_student = train_distilled_classifier(
    STUDENT_ENCODER_ID, soft_train, splits.gold_val, num_labels=N, temperature=2.0, alpha=0.5
)
soft_pred = predict(soft_student, splits.test.texts)
soft_result = summarize(splits.test.labels, soft_pred, N)
save_result("distilled_probabilities_T2", soft_result, soft_pred)
print(soft_result)

## 6. Compare

In [ ]:
from kd_course.metrics import accuracy, gap_closed, paired_bootstrap
from kd_course.results import format_results, load_results

print(format_results(load_results()))
print()

base_acc = baseline_result["metrics"]["accuracy"]
for name, result in (("Top answers", hard_result), ("Probabilities", soft_result)):
    gap = gap_closed(base_acc, result["accuracy"], teacher_result["accuracy"])
    text = "no gap to close" if gap is None else f"{gap:.0%} of the gap closed"
    print(f"{name:<14} {text}")

In [ ]:
def report(title, a, b):
    r = paired_bootstrap(accuracy, splits.test.labels, a, b)
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


report("Probabilities minus baseline   ", baseline_result["predictions"], soft_pred)
report("Probabilities minus top answers", hard_pred, soft_pred)

## 7. Choose a temperature

Train with two more temperatures and compare them **on the validation set**. The best validation
accuracy for each run is in its training history. Do not look at test scores to choose.

In [ ]:
del soft_student
gc.collect()
torch.cuda.empty_cache()

best_val = {2.0: None}
for temperature in (1.0, 4.0):
    student = train_distilled_classifier(
        STUDENT_ENCODER_ID,
        soft_train,
        splits.gold_val,
        num_labels=N,
        temperature=temperature,
        alpha=0.5,
    )
    best_val[temperature] = max(row["val_accuracy"] for row in student.history)
    del student
    gc.collect()
    torch.cuda.empty_cache()

print({t: v for t, v in best_val.items() if v is not None})

## 8. What did you see?

Double-click this cell and write your answers.

**a. How much of the gap did the probability-trained student close? Is its gain over the baseline real? Quote the interval.**

**b. Did probabilities beat top answers? Is that difference real? What does your answer say about where the teacher's value lies for this task?**

**c. Which temperature had the best validation accuracy? With only 308 validation messages, how sure can you be of that choice? (Lesson 2.4 has the tool for this.)**

**d. The teacher's score here may differ from its Lab 2 score. Why?**